# SQL Murder Mystery — Investigación

Resolución del caso de asesinato ocurrido el 15 de enero de 2018 en SQL City,
usando la base de datos `data/sql-murder-mystery.db`.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/sql-murder-mystery.db")  # ajusta la ruta relativa según dónde guardes el notebook

## 1. La escena del crimen

Filtramos por los datos iniciales del enunciado: ciudad SQL City, fecha 15 de enero de 2018, tipo asesinato.

In [ ]:
query = """
SELECT description
FROM crime_scene_report
WHERE city = 'SQL City'
  AND type = 'murder'
  AND date = 20180115;
"""
pd.read_sql_query(query, conn)

,description
0,Security footage shows that there were 2 witne...


**Resultado:** hay dos testigos. El primero vive en la última casa de "Northwestern Dr".
La segunda, Annabel, vive en "Franklin Ave".

## 2. Identificar y entrevistar a los testigos

Cruzamos `person` (datos de residencia) con `interview` (declaraciones) por su identificador,
filtrando por la calle del primer testigo o por el nombre y calle de Annabel.

In [4]:
query = """
SELECT p.name, i.transcript, p.id
FROM person p
JOIN interview i ON p.id = i.person_id
WHERE (p.address_street_name = 'Northwestern Dr')
   OR (p.name LIKE 'Annabel%' AND p.address_street_name = 'Franklin Ave');
"""
pd.read_sql_query(query, conn)

,name,transcript,id
0,Teri Ehrich,"sea, some children digging in the sand with wo...",88423
1,Vincenza Burkhardt,"Poor Alice! It was as much as she could do, ly...",34352
2,Weldon Penso,the verses to himself: ‘“WE KNOW IT TO BE TRUE...,15171
3,Coretta Cubie,"head in the lap of her sister, who was gently ...",96595
4,Courtney Bordeaux,"see: four times five is twelve, and four times...",72076
5,Rashad Cascone,"for apples, yer honour!’\n",28360
6,Del Tacderen,"‘We had the best of educations--in fact, we we...",75484
7,Olevia Morena,when I learn music.’\n,25615
8,Angelena Billman,"Will you, won’t you, will you, won’t you, won...",26758
9,Abe Roeker,"‘Hold your tongue!’ said the Queen, turning pu...",39688


**Resultado clave:**
- Annabel Miller: *"Vi cómo se cometía el asesinato y reconocí al asesino de mi gimnasio, donde
  estuve entrenando la semana pasada, el 9 de enero."*
- Morty Schapiro (uno de los resultados de Northwestern Dr): *"Oí un disparo y vi a un hombre
  salir corriendo. Llevaba una bolsa del gimnasio 'Get Fit Now'. El número de socio empezaba
  por '48Z'. Solo los socios Gold tienen esas bolsas. Se subió a un coche con matrícula que
  incluía 'H42W'."*

## 3. Filtrar sospechosos por el gimnasio

In [5]:
query = """
SELECT *
FROM get_fit_now_member
WHERE id LIKE '48Z%'
  AND membership_status = 'gold'
LIMIT 20;
"""
pd.read_sql_query(query, conn)

,id,person_id,name,membership_start_date,membership_status
0,48Z7A,28819,Joe Germuska,20160305,gold
1,48Z55,67318,Jeremy Bowers,20160101,gold


## 4. Cruzar con la matrícula del coche

In [6]:
query = """
SELECT p.name, dl.*, m.*
FROM person p
JOIN drivers_license dl ON p.license_id = dl.id
JOIN get_fit_now_member m ON p.id = m.person_id
WHERE m.id LIKE '48Z%'
  AND dl.plate_number LIKE '%H42W%';
"""
pd.read_sql_query(query, conn)

,name,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,id,person_id,name,membership_start_date,membership_status
0,Jeremy Bowers,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS,48Z55,67318,Jeremy Bowers,20160101,gold


**Resultado:** el asesino material es **Jeremy Bowers**.

## 5. Verificar el sospechoso

In [7]:
query = """
INSERT INTO solution VALUES (1, 'Jeremy Bowers');
SELECT value FROM solution;
"""
conn.executescript(query)
pd.read_sql_query("SELECT value FROM solution;", conn)

,value
0,"Congrats, you found the murderer! But wait, th..."


El juego confirma que Jeremy Bowers es el asesino, pero indica que fue contratado por otra
persona y pide identificar al cerebro detrás del crimen a partir de su testimonio.

## 6. El testimonio del asesino: buscando al cerebro

In [8]:
query = """
SELECT transcript
FROM interview
WHERE person_id = (SELECT id FROM person WHERE name = 'Jeremy Bowers');
"""
pd.read_sql_query(query, conn)

,transcript
0,I was hired by a woman with a lot of money. I ...


**Resultado:** *"Me contrató una mujer muy rica. No sé su nombre, pero mide entre 5'5" (65") y
5'7" (67"). Tiene el pelo rojo y conduce un Tesla Model S. Asistió tres veces al SQL Symphony
Concert en diciembre de 2017."*

Cruzamos `drivers_license` (color de pelo, marca y modelo de coche) con
`facebook_event_checkin` (evento y fecha) para encontrarla.

In [9]:
query = """
SELECT p.name
FROM person p
JOIN drivers_license dl ON p.license_id = dl.id
JOIN facebook_event_checkin fec ON p.id = fec.person_id
WHERE dl.hair_color = 'red'
  AND dl.car_make = 'Tesla'
  AND dl.car_model = 'Model S'
  AND fec.event_name = 'SQL Symphony Concert'
  AND fec.date LIKE '201712%';
"""
pd.read_sql_query(query, conn)

,name
0,Miranda Priestly
1,Miranda Priestly
2,Miranda Priestly


**Resultado:** la instigadora del crimen es **Miranda Priestly**.

## 7. Verificación final

In [10]:
query = """
INSERT INTO solution VALUES (1, 'Miranda Priestly');
SELECT value FROM solution;
"""
conn.executescript(query)
pd.read_sql_query("SELECT value FROM solution;", conn)

,value
0,"Congrats, you found the brains behind the murd..."


## Conclusión

La investigación arrancó a partir del informe de la escena del crimen (`crime_scene_report`),
que apuntaba a un asesinato ocurrido el 15 de enero de 2018 en SQL City y mencionaba a dos
testigos: uno en la última casa de Northwestern Dr, y Annabel, en Franklin Ave.

Cruzando las tablas `person` e `interview` obtuvimos dos declaraciones clave:

- **Annabel Miller** reconoció al asesino de su propio gimnasio, donde había coincidido con
  él entrenando el 9 de enero.
- **Morty Schapiro** (uno de los vecinos de Northwestern Dr) aportó tres pistas físicas
  sobre el agresor: llevaba una bolsa del gimnasio "Get Fit Now", su número de socio
  empezaba por "48Z" (exclusivo de socios *Gold*), y huyó en un coche con una matrícula
  que incluía "H42W".

Con esas pistas filtramos `get_fit_now_member` por el prefijo de socio y el estado Gold,
lo que dejó dos sospechosos, y usamos `drivers_license` para cruzar la matrícula parcial.
Uniendo ambas tablas a través de `person`, solo una persona cumplía las dos condiciones a
la vez: **Jeremy Bowers**, confirmado como el asesino material al insertarlo en la tabla
`solution`.

Sin embargo, el propio juego reveló que Bowers fue solo el ejecutor: alguien lo contrató.
Su testimonio en `interview` describió a una mujer con recursos económicos, de entre 5'5"
y 5'7" de altura, pelirroja, conductora de un Tesla Model S, que había asistido tres veces
al SQL Symphony Concert en diciembre de 2017. Cruzando `drivers_license` (color de pelo,
marca y modelo del coche) con `facebook_event_checkin` (evento y fecha), apareció un único
nombre que cumplía todos los criterios: **Miranda Priestly**, confirmada como la instigadora
del crimen al verificarla en `solution`.

**Resumen del caso:** Jeremy Bowers asesinó por encargo, contratado y planeado por Miranda
Priestly. La resolución combinó filtrado por texto y fecha, JOINs entre múltiples tablas
(`person`, `interview`, `get_fit_now_member`, `drivers_license`, `facebook_event_checkin`)
y verificación final contra la tabla `solution`, siguiendo el rastro de pista en pista sin
ninguna suposición no respaldada por los datos.